In [ ]:
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import itertools
from pylab import *
from skimage import data
from skimage.viewer.canvastools import RectangleTool
from skimage.viewer import ImageViewer
from tqdm import tqdm
import torch
import torch.nn as nn  
import torch.optim as optim 
import torchvision.transforms as transforms 
import torch.nn.functional as F
import torchvision
import pandas as pd
from skimage import io
from torch.utils.data import (
    Dataset,
    DataLoader,
)
from PIL import Image
import math
import json
%pip install pybboxes
%pip install patchify

In [ ]:
# import zipfile
# with zipfile.ZipFile('./data.zip', 'r') as zip_ref:
#     zip_ref.extractall('./dataset')
#!wget https://github.com/ch-hristov/p-id-symbols/raw/main/model.py --quiet

## Build a class model for the data. 


This data can be used to test how well our models perform

In [ ]:
# Move the data into patches
import patchify


def tile(filename, dir_in, dir_out, d):

    if os.path.exists(dir_out) is False:
        os.mkdir(dir_out)

    name, ext = os.path.splitext(filename)
    img = Image.open(os.path.join(dir_in, filename))
    image = np.array(img)

    overlap = int(d/2)
    patches = patchify.patchify(image, (d, d, 3), step=overlap)

    for i in range(patches.shape[0]):
        for j in range(patches.shape[1]):
            single_patch_img = patches[i, j, 0, :, :, :]
            img = Image.fromarray(single_patch_img)
            img.save('{0}_{1}_{2}.jpg'.format(dir_out + "/" +
                     filename.replace('.jpg', ''), int(i*d/2), int(j*d/2)))


if not os.path.exists('./images'):
    for i in range(0, 500):
        tile("{0}.jpg".format(str(i)),
             './dataset/real-life-images', './images', 1280)


## Iterate through the directory and load the information encoded in .npy files

This information is made available by the DIGITIZE-PID research.

## Individual dataset building
1. Symbols - group by symbol label.

In [8]:
import model
import pid_parser as pid

from importlib import reload

npy_path = "./dataset"
patch_resolution = 1280

reload(model)
reload(pid)

parser = pid.PIDParser()


pids = parser.parse_folder_npy(npy_path)

lines: dict[str, list[model.VisualObject]] = {}
symbols:  dict[str, list[model.VisualObject]] = {}

for pid_item in pids:
    for line in pid_item.lines:
        if line.pid.id not in lines:
            lines[line.pid.id] : list[model.VisualObject] = []
        lines[line.pid.id].append(line)


for pid_item in pids:
    for obj in pid_item.symbols:
        if obj.pid.id not in symbols:
            symbols[obj.pid.id] : list[model.VisualObject] = []
        symbols[obj.pid.id].append(obj)

print(parser.generate_line_labels(
    pids, './line_images', './line_labels', lines))


KeyboardInterrupt: 

In [24]:
q = ([i for i in pids if i.id == '111'])[0].lines
line = ([line for line in q if line.line_id == 'line_1'])[0]

print([line.start_xy, line.end_xy, line.types])

[[3832, 999], [3639, 999], 'solid']


## Split the image into equal sized patches

In [ ]:
from itertools import product
import patchify
import os

def train_test():
    import glob
    import os
    import numpy as np
    import sys
    current_dir = "./images"
    split_pct = 10  # 10% validation set
    file_train = open("./train_line.txt", "w")  
    file_val = open("./val_line.txt", "w")  
    
    counter = 1  
    index_test = round(100 / split_pct)  
    for fullpath in glob.iglob(os.path.join(current_dir, "*.jpg")):  
      title, ext = os.path.splitext(os.path.basename(fullpath))
      if counter == index_test:
        counter = 1
        file_val.write(current_dir + "/" + title + '.jpg' + "\n")
      else:
        file_train.write(current_dir + "/" + title + '.jpg' + "\n")
        counter = counter + 1
    file_train.close()
    file_val.close()

train_test()

In [ ]:
import os, shutil

folder = './backup/.ipynb_checkpoints'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))
        
import os, shutil
folder = './labels/.ipynb_checkpoints'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os.path
import pybboxes
CIS = 1280

def map_labels():
    all_images = os.listdir("./images")
    coords = []
    rendered = False
    
    for fn in all_images:
        if not fn.endswith('.jpg'):
            continue
        
        coords = []
        if 'ipynb_checkpoints' in fn:
            continue
        
        original_image_id = fn.split('_')[0]
        # original labels
        label_target = './symbol_labels/{0}.txt'.format(original_image_id.replace('.jpg',''))
        output_path = './labels/{0}'.format(fn.replace('.jpg', '.txt'))

        if os.path.exists('./labels') is False:
         os.mkdir('./labels')
        
        matched_pid = None
        
        for pid in pids:
            if(pid.id == original_image_id):
                matched_pid = pid
                break
                
        spl = fn.split('_')
        f = float(spl[2].replace('.jpg',""))
        
        img_x_start, img_y_start = float(f), float(spl[1])
        img_x_end, img_y_end = f + CIS, float(spl[1]) + CIS
                
        list_lines, final = [], []
        
        with open(label_target) as file:
            lines = [line.rstrip() for line in file]
            
            for line in lines:
                _class, x, y, x1, y1 = line.split(' ')
                
                x_real, y_real = float(x) * pid.width, float(y) * pid.height
                x1_real, y1_real = float(x1) * pid.width, float(y1) * pid.height
                
                _add = [x_real, y_real, x1_real, y1_real, _class]
                
                coords.append(_add)
                
                 # we know that this symbol is on this image after this line
                if x_real>= img_x_start and x1_real <= img_x_end and y_real >= img_y_start and y1_real <= img_y_end:
                    
                    x_img_real = x_real - img_x_start
                    y_img_real = y_real - img_y_start
                    
                    x1_img_real = x1_real - img_x_start
                    y1_img_real = y1_real - img_y_start
                    
                    if x_img_real < 0:
                        x_img_real = 0
                        continue
                    
                    if(y_img_real < 0):
                        y_img_real = 0
                        continue
                    
                    if(x1_img_real < 0):
                        x1_img_real = 0
                        continue
                    
                    if(y1_img_real < 0):
                        y1_img_real = 0
                        continue
                        
                    if(x_img_real >= CIS):
                        x_img_real = CIS
                        continue
                        
                    if(y_img_real >= CIS):
                        y_img_real = CIS
                        continue
                        
                    if(x1_img_real >= CIS):
                        x1_img_real = CIS
                        continue
                        
                    if(y1_img_real >= CIS):
                        y1_img_real = CIS
                        continue
                    
                    min_x = min(x_img_real, x1_img_real)
                    min_y = min(y_img_real, y1_img_real)
                    
                    max_x =  max(x_img_real, x1_img_real)
                    max_y = max(y_img_real, y1_img_real)
                    
                    if(max_x >= CIS):
                        max_x = CIS
                        min_x = min_x - 1
                        continue
                        
                    if(max_y >= CIS):
                        max_y = CIS
                        min_y = min_y - 1
                        continue
                        
                    voc_bbox = (min_x, min_y , max_x, max_y)
                    
                    result = pybboxes.convert_bbox(voc_bbox, from_type="voc", to_type="yolo", image_size=(CIS,CIS))
                    
                    _str = "{0} {1} {2} {3} {4}\r\n".format(int(_class), *result)
                    list_lines.append(_str)
                    

            with open(output_path, 'w+') as f:
                f.writelines(list_lines)
                        
        if (original_image_id == '1' and rendered is False):
            rendered = True
            im = Image.open(matched_pid.path)
            fig, ax = plt.subplots()
            fig.set_size_inches(30, 30)
            ax.imshow(im)
            
            rectangles = []
            
            # for coord in coords:
            #     rect = patches.Rectangle((coord[0],coord[1]), coord[2]-coord[0], coord[3]-coord[1], 
            #                              linewidth=1, label = coord[4])
            
            #     ax.add_artist(rect)
            #     rx, ry = rect.get_xy()
            #     cx = rx + rect.get_width()
            #     cy = ry + rect.get_height()

            #     ax.annotate(coord[4], (cx, cy), color='blue', weight='bold', 
            #                 fontsize=20)
                
            ax.set_aspect('equal')
            plt.show()

map_labels()

## Map the labels to individual annotated patches

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os.path
import pybboxes

def map_labels():
    all_images = os.listdir("./images")
    coords = []
    rendered = False
    
    for fn in all_images:
        #print(fn)
        if not fn.endswith('.jpg'):
            continue
        
        coords = []
        if 'ipynb_checkpoints' in fn:
            continue
        
        original_image_id = fn.split('_')[0]
        # original labels
        label_target = './label_items_lines/{0}.txt'.format(original_image_id.replace('.jpg',''))
        output_path = './labels/{0}'.format(fn.replace('.jpg', '.txt'))
        
        matched_pid = None
        
        for pid in pids:
            if(pid.id == original_image_id):
                matched_pid = pid
                break
                
        spl = fn.split('_')
        f = float(spl[2].replace('.jpg',""))
        
        img_x_start, img_y_start = float(f), float(spl[1])
        img_x_end, img_y_end = f + CIS, float(spl[1]) + CIS
                
        list_lines, final = [], []
        
        with open(label_target) as file:
            lines = [line.rstrip() for line in file]
            
            for line in lines:
                _class, x, y, x1, y1 = line.split(' ')
                
                x_real, y_real = float(x) * pid.width, float(y) * pid.height
                x1_real, y1_real = float(x1) * pid.width, float(y1) * pid.height
                
                _add = [x_real, y_real, x1_real, y1_real, _class]
                
                coords.append(_add)
                
                 # we know that this symbol is on this image after this line
                if x_real>= img_x_start and x1_real <= img_x_end and y_real >= img_y_start and y1_real <= img_y_end:
                    
                    x_img_real = x_real - img_x_start
                    y_img_real = y_real - img_y_start
                    
                    x1_img_real = x1_real - img_x_start
                    y1_img_real = y1_real - img_y_start
                    
                    if x_img_real < 0:
                        x_img_real = 0
                        continue
                    
                    if(y_img_real < 0):
                        y_img_real = 0
                        continue
                    
                    if(x1_img_real < 0):
                        x1_img_real = 0
                        continue
                    
                    if(y1_img_real < 0):
                        y1_img_real = 0
                        continue
                        
                        
                    if(x_img_real >= CIS):
                        x_img_real = CIS
                        continue
                        
                    if(y_img_real >= CIS):
                        y_img_real = CIS
                        continue
                        
                    if(x1_img_real >= CIS):
                        x1_img_real = CIS
                        continue
                        
                    if(y1_img_real >= CIS):
                        y1_img_real = CIS
                        continue
                    
                    min_x = min(x_img_real, x1_img_real)
                    min_y = min(y_img_real, y1_img_real)
                    
                    max_x =  max(x_img_real, x1_img_real) + 10
                    max_y = max(y_img_real, y1_img_real) + 10
                    
                    if(max_x >= CIS):
                        max_x = CIS
                        min_x = min_x - 1
                        
                        
                    if(max_y >= CIS):
                        max_y = CIS
                        min_y = min_y - 1
                        
                    
                    voc_bbox = (min_x, min_y , max_x, max_y)
                    
                    result = pybboxes.convert_bbox(voc_bbox, from_type="voc", to_type="yolo", image_size=(CIS,CIS))
                    
                    _str = "{0} {1} {2} {3} {4}\r\n".format(int(_class), *result)
                    list_lines.append(_str)
                    
                    if original_image_id == '1' and int(img_x_start) == 0 and int(img_x_end) == pid.width and int(img_y_start)== 0 and int(img_y_end) == pid.height:
                        fig, ax = plt.subplots()
                        fig.set_size_inches(5, 5)
                        
                        rect = patches.Rectangle((voc_bbox[0],voc_bbox[1]), voc_bbox[2]-voc_bbox[0], voc_bbox[3]-voc_bbox[1],
                                         linewidth=1, label = _class)
                        
                        ax.add_artist(rect)
                        rx, ry = rect.get_xy()
                        
                        cx = rx + rect.get_width()
                        cy = ry + rect.get_height()

                        ax.annotate(_class, (cx, cy), color='w', weight='bold', 
                                    fontsize=10, ha='center', va='center')
                        
                        img = Image.open('./images/' + fn)
                        ax.imshow(img)
                        plt.show()

            with open(output_path, 'w+') as f:
                f.writelines(list_lines)
                        
        if (original_image_id == '1' and rendered is False):
            rendered = True
            im = Image.open(matched_pid.path)
            fig, ax = plt.subplots()
            fig.set_size_inches(30, 30)
            ax.imshow(im)
            
            rectangles = []
            
            for coord in coords:
                rect = patches.Rectangle((coord[0],coord[1]), coord[2]-coord[0], coord[3]-coord[1], 
                                         linewidth=1, label = coord[4])
            
                ax.add_artist(rect)
                rx, ry = rect.get_xy()
                cx = rx + rect.get_width()
                cy = ry + rect.get_height()

                ax.annotate(coord[4], (cx, cy), color='blue', weight='bold', 
                            fontsize=20)
                
            ax.set_aspect('equal')
            plt.show()

map_labels()

In [ ]:
from distutils.dir_util import copy_tree
import shutil
shutil.make_archive('images', 'zip', './images')
shutil.make_archive('labels', 'zip', './labels')
shutil.make_archive('original_imgs', 'zip', './sample_data')
shutil.make_archive('original_labels', 'zip', './symbol_labels')

In [ ]:
shutil.move("/content/images.zip", "/content/drive/MyDrive/symbol_extract")
shutil.move("/content/labels.zip", "/content/drive/MyDrive/symbol_extract")
shutil.move("/content/train.txt", "/content/drive/MyDrive/symbol_extract")
shutil.move("/content/val.txt", "/content/drive/MyDrive/symbol_extract")

In [ ]:
!pip install patool
import patoolib
#patoolib.extract_archive('/content/drive/MyDrive/symbol_extract/labels.zip', outdir='/content/drive/MyDrive/symbol_extract/labels')
patoolib.extract_archive('/content/drive/MyDrive/symbol_extract/images.zip', outdir='/content/drive/MyDrive/symbol_extract/image_targets')

In [ ]:
import shutil
im = 'content/drive/MyDrive/symbol_extract/images'
shutil.rmtree(im)